<a href="https://colab.research.google.com/github/Machine-Learning-Visao-Computacional-T1/CodigoDosAlunos/blob/main/exemploconexaosqlpython.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install psycopg2-binary SQLAlchemy pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 43.9 MB/s eta 0:00:00


In [17]:
import pandas as pd
from sqlalchemy import create_engine

host = "dpg-d8srl9kmmk8c73dmkdn0-a.virginia-postgres.render.com"
port = "5432"
dbname = "sctech_db"
user = "sctech_db_user"
password = "R3Xbz4qUy52B3CcM756RQSD2oiAyzwDm"

string_conexao = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
engine = create_engine(string_conexao)

query_descobrir_tabelas = """
SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_NAME = 'carros';
"""

df_tabelas = pd.read_sql(query_descobrir_tabelas, engine)

engine.dispose()

print("Tabelas disponíveis nesta base de dados:")
display(df_tabelas)

Tabelas disponíveis nesta base de dados:


,column_name,data_type,character_maximum_length
0,preco,numeric,NaN
1,ano,integer,NaN
2,quilometragem,integer,NaN
3,potencia_cv,integer,NaN
4,id,integer,NaN
5,marca,character varying,50.0
6,modelo,character varying,50.0
7,tipo_combustivel,character varying,20.0
8,transmissao,character varying,20.0


In [19]:
query = "SELECT * FROM carros"

df_carros = pd.read_sql(query, engine)

engine.dispose()

display(df_carros.head())


def padronizar_colunas(df):
    df = df.copy()
    df.columns = [col.lower().replace(' ', '_') for col in df.columns]
    return df

def tratar_nulos(df):
    df = df.copy()
    df = df.dropna()
    return df

def remover_duplicados(df):
    df = df.copy()
    df = df.drop_duplicates()
    return df


#df_tratado = (df_carros
 #             .pipe(padronizar_colunas)
  #            .pipe(tratar_nulos)
   #           .pipe(remover_duplicados)
    #         )

print("\nPipeline concluído:")
display(df_carros.info())

,id,marca,modelo,ano,quilometragem,tipo_combustivel,transmissao,potencia_cv,preco
0,1,Nissan,Cupê,2000,43897,Gasolina,Manual,312,25750.77
1,2,Volkswagen,Sedã,2006,238211,Elétrico,Automático,244,23267.02
2,3,Kia,SUV,2009,278428,Diesel,Automático,269,27402.47
3,4,Chevrolet,SUV,2016,208617,Flex,Manual,194,37729.01
4,5,Ford,Cupê,2015,1320,Híbrido,Manual,182,39929.59



Pipeline concluído:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                1000 non-null   int64  
 1   marca             1000 non-null   object 
 2   modelo            1000 non-null   object 
 3   ano               1000 non-null   int64  
 4   quilometragem     1000 non-null   int64  
 5   tipo_combustivel  1000 non-null   object 
 6   transmissao       1000 non-null   object 
 7   potencia_cv       1000 non-null   int64  
 8   preco             1000 non-null   float64
dtypes: float64(1), int64(4), object(4)
memory usage: 70.4+ KB


None

In [13]:
df_tratado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                1000 non-null   int64  
 1   marca             1000 non-null   object 
 2   modelo            1000 non-null   object 
 3   ano               1000 non-null   int64  
 4   quilometragem     1000 non-null   int64  
 5   tipo_combustivel  1000 non-null   object 
 6   transmissao       1000 non-null   object 
 7   potencia_cv       1000 non-null   int64  
 8   preco             1000 non-null   float64
dtypes: float64(1), int64(4), object(4)
memory usage: 70.4+ KB


In [16]:
import psycopg2
import io

conn = psycopg2.connect(
    host="dpg-d8srl9kmmk8c73dmkdn0-a.virginia-postgres.render.com", database="sctech_db", user="sctech_db_user", password="R3Xbz4qUy52B3CcM756RQSD2oiAyzwDm"
)

table_name = "carros"

try:
    with conn.cursor() as cur:
        # 3. Truncate the existing table
        cur.execute(f"TRUNCATE TABLE {table_name};")

        # 4. Prepare data in memory as a text stream
        output = io.StringIO()
        df_tratado.to_csv(output, sep="\t", header=False, index=False)
        output.seek(0)

        # 5. Bulk insert using COPY
        cur.copy_from(output, table_name, sep="\t", columns=list(df_tratado.columns))

    # Commit both truncate and insert inside the same transaction block
    conn.commit()
    print("Table truncated and updated successfully.")

except Exception as e:
    conn.rollback()
    print(f"An error occurred: {e}")
finally:
    conn.close()

Table truncated and updated successfully.
